# NES Data Cleaning Notebook (Template)

This notebook is focused on:
- missing values diagnostics (counts and % per column),
- practical missing-data cleanup,
- robust outlier detection/removal across numeric columns.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import os
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from matplotlib.font_manager import fontManager, FontProperties

DATA_PATH = "/Users/royalmustaches/Documents/Programming/Dano/DANO_ITMO_2026/Datasets/Base_nes.csv"

PALETTE = ["#5e17eb", "#ff6b6b", "#00c49a"]
FONT_CANDIDATES = [
    "/Users/royalmustaches/Documents/Programming/Dano/DANO_ITMO_2026/HSE_FONTS_FOR_GRAPHS_copy/HSESlab-Regular.ttf",
    "/Users/royalmustaches/Documents/Programming/Dano/DANO_NES_2026/HSE_FONTS_FOR_GRAPHS/HSESlab-Regular.ttf",
    "/Users/royalmustaches/Documents/Programming/Dano/Practice/Organization/Fonts_for_projects/HSESlab-Regular.ttf",
]

FONT_NAME = "DejaVu Sans"
for font_path in FONT_CANDIDATES:
    if os.path.exists(font_path):
        fontManager.addfont(font_path)
        FONT_NAME = FontProperties(fname=font_path).get_name()
        break

sns.set_theme(style="whitegrid", font=FONT_NAME, context="notebook", palette=PALETTE)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3
plt.rcParams["grid.linestyle"] = "--"

PRIMARY = "#5e17eb"
SECONDARY = "#ff6b6b"
ACCENT = "#00c49a"

In [ ]:
df = pd.read_csv(DATA_PATH)
print("Shape:", df.shape)
print("Font in use:", FONT_NAME)

df.head(3)

## 1) Initial Quality Snapshot

In [ ]:
quality_overview = pd.DataFrame({
    "column": df.columns,
    "dtype": df.dtypes.astype(str).values,
    "n_unique": [df[c].nunique(dropna=True) for c in df.columns],
    "missing_count": [df[c].isna().sum() for c in df.columns],
    "missing_pct": [round(df[c].isna().mean() * 100, 2) for c in df.columns],
})

quality_overview = quality_overview.sort_values(["missing_pct", "n_unique"], ascending=[False, False]).reset_index(drop=True)
quality_overview.head(30)

In [ ]:
# Full table of missing values by column
missing_table = pd.DataFrame({
    "column": df.columns,
    "missing_count": df.isna().sum().values,
    "missing_pct": (df.isna().mean().values * 100).round(2),
}).sort_values("missing_pct", ascending=False).reset_index(drop=True)

missing_table

In [ ]:
# Missingness summary buckets
missing_bins = pd.cut(
    missing_table["missing_pct"],
    bins=[-0.01, 0, 5, 20, 50, 100],
    labels=["0%", "0-5%", "5-20%", "20-50%", "50-100%"],
)
missing_bucket_table = missing_bins.value_counts().sort_index().rename_axis("missing_bucket").reset_index(name="num_columns")
missing_bucket_table

In [ ]:
# Visualization: top 25 columns by missing %
plot_missing = missing_table.head(25)

plt.figure(figsize=(12, 7))
sns.barplot(data=plot_missing, y="column", x="missing_pct", color=PRIMARY)
plt.title("Top 25 columns by missing values (%)")
plt.xlabel("Missing, %")
plt.ylabel("")
plt.tight_layout()
plt.show()

## 2) Missing Values Handling Helpers

In [ ]:
def drop_columns_by_missing(df_in, threshold_pct=70.0):
    """
    Drop columns with missing percentage above threshold.
    """
    miss_pct = df_in.isna().mean() * 100
    cols_to_drop = miss_pct[miss_pct > threshold_pct].index.tolist()
    return df_in.drop(columns=cols_to_drop), cols_to_drop


def fill_missing_basic(df_in):
    """
    Basic baseline imputation:
    - numeric -> median
    - object/category -> mode (or 'Unknown' if mode absent)
    """
    out = df_in.copy()
    for col in out.columns:
        if pd.api.types.is_numeric_dtype(out[col]):
            med = out[col].median()
            out[col] = out[col].fillna(med)
        else:
            mode = out[col].mode(dropna=True)
            fill_value = mode.iloc[0] if len(mode) else "Unknown"
            out[col] = out[col].fillna(fill_value)
    return out

In [ ]:
# Example: drop highly-missing columns then fill rest
work_df, dropped_cols = drop_columns_by_missing(df, threshold_pct=70.0)
work_df = fill_missing_basic(work_df)

print("Dropped columns (>70% missing):", len(dropped_cols))
print(dropped_cols[:15])
print("Remaining shape:", work_df.shape)

## 3) Outlier Detection and Deletion (Across Columns)

In [ ]:
def select_numeric_for_outliers(df_in, min_unique=10, exclude_prefixes=("id_",)):
    """
    Select numeric columns suitable for outlier filtering.
    Excludes likely ID columns and low-cardinality binary flags.
    """
    cols = []
    for col in df_in.select_dtypes(include=[np.number]).columns:
        if any(col.startswith(p) for p in exclude_prefixes):
            continue
        nunique = df_in[col].nunique(dropna=True)
        if nunique >= min_unique:
            cols.append(col)
    return cols


def outlier_report_iqr(df_in, cols, k=1.5):
    """
    Per-column outlier counts using IQR rule.
    """
    rows = []
    n = len(df_in)
    for col in cols:
        s = pd.to_numeric(df_in[col], errors="coerce")
        q1, q3 = s.quantile(0.25), s.quantile(0.75)
        iqr = q3 - q1
        if pd.isna(iqr) or iqr == 0:
            lower, upper = q1, q3
            out_n = 0
        else:
            lower, upper = q1 - k * iqr, q3 + k * iqr
            out_n = int(((s < lower) | (s > upper)).sum())
        rows.append({
            "column": col,
            "q1": q1,
            "q3": q3,
            "lower": lower,
            "upper": upper,
            "outliers_n": out_n,
            "outliers_pct": round(out_n / n * 100, 3),
        })
    return pd.DataFrame(rows).sort_values("outliers_pct", ascending=False).reset_index(drop=True)


def delete_outliers_iqr_across_columns(df_in, cols, k=1.5):
    """
    Remove rows that are outliers in ANY of selected columns (IQR rule).
    """
    keep_mask = pd.Series(True, index=df_in.index)

    for col in cols:
        s = pd.to_numeric(df_in[col], errors="coerce")
        q1, q3 = s.quantile(0.25), s.quantile(0.75)
        iqr = q3 - q1
        if pd.isna(iqr) or iqr == 0:
            continue
        lower, upper = q1 - k * iqr, q3 + k * iqr
        # keep NaN rows unchanged; filter only defined outliers
        col_mask = s.isna() | ((s >= lower) & (s <= upper))
        keep_mask &= col_mask

    cleaned = df_in.loc[keep_mask].copy()
    dropped_idx = df_in.index[~keep_mask]
    summary = {
        "rows_before": int(len(df_in)),
        "rows_after": int(len(cleaned)),
        "rows_removed": int((~keep_mask).sum()),
        "rows_removed_pct": round((~keep_mask).mean() * 100, 3),
    }
    return cleaned, dropped_idx, summary

In [ ]:
# Choose columns and inspect outliers
candidate_cols = select_numeric_for_outliers(work_df, min_unique=10)
outlier_table_before = outlier_report_iqr(work_df, candidate_cols, k=1.5)
outlier_table_before.head(20)

In [ ]:
# Apply deletion of outliers across selected columns
clean_df, removed_idx, outlier_summary = delete_outliers_iqr_across_columns(
    work_df,
    cols=candidate_cols,
    k=1.5
)

outlier_summary

In [ ]:
# Compare before/after for key numeric columns
key_numeric = [c for c in ["salary", "experience", "birthday", "inner_info_fullness_rate"] if c in clean_df.columns]

before_stats = work_df[key_numeric].describe().T[["mean", "std", "min", "25%", "50%", "75%", "max"]]
after_stats = clean_df[key_numeric].describe().T[["mean", "std", "min", "25%", "50%", "75%", "max"]]

compare_stats = before_stats.join(after_stats, lsuffix="_before", rsuffix="_after")
compare_stats

In [ ]:
# Optional: visualize distribution changes after outlier deletion
for col in [c for c in ["salary", "experience"] if c in clean_df.columns]:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
    sns.histplot(work_df[col], bins=50, kde=True, ax=axes[0], color=SECONDARY)
    axes[0].set_title(f"Before cleaning: {col}")

    sns.histplot(clean_df[col], bins=50, kde=True, ax=axes[1], color=PRIMARY)
    axes[1].set_title(f"After cleaning: {col}")

    plt.tight_layout()
    plt.show()

## 4) Save Clean Dataset

In [ ]:
OUTPUT_PATH = "/Users/royalmustaches/Documents/Programming/Dano/DANO_ITMO_2026/Datasets/Base_nes_cleaned.csv"
clean_df.to_csv(OUTPUT_PATH, index=False)
print("Saved:", OUTPUT_PATH)
print("Final shape:", clean_df.shape)